# Modelo para tipo de ingreso (Mas categorias)

In [1]:
import pandas as pd
import random

random.seed(42)

datos = []

# =====================================
# LISTAS GENERALES
# =====================================

acciones = [
    "Pago",
    "Abono",
    "Transferencia",
    "Depósito",
    "Consignación",
    "Ingreso",
    "Recepción",
    "Cobro"
]

bancos = [
    "",
    "Bancolombia",
    "Davivienda",
    "BBVA",
    "Banco de Bogotá",
    "Banco Popular",
    "Nequi",
    "Daviplata"
]

meses = [
    "",
    "enero",
    "febrero",
    "marzo",
    "abril",
    "mayo",
    "junio",
    "julio",
    "agosto",
    "septiembre",
    "octubre",
    "noviembre",
    "diciembre"
]

empresas = [
    "",
    "Empresa ABC",
    "Empresa XYZ",
    "Google",
    "Oracle",
    "Microsoft",
    "Universidad",
    "Cliente",
    "Tienda",
    "Startup"
]

# =====================================
# CATEGORÍAS
# =====================================

categorias = {

    "Salario":[
        "salario",
        "sueldo",
        "nómina",
        "nomina",
        "remuneración",
        "bonificación",
        "prima",
        "cesantías",
        "liquidación",
        "horas extras"
    ],

    "Honorarios":[
        "honorarios",
        "consultoría",
        "consultoria",
        "servicios profesionales",
        "prestación de servicios",
        "trabajo freelance",
        "proyecto freelance",
        "asesoría",
        "asesoria"
    ],

    "Negocio":[
        "ventas",
        "cliente",
        "factura",
        "pedido",
        "emprendimiento",
        "negocio",
        "productos",
        "servicios",
        "comisión",
        "comision"
    ],

    "Rendimiento_inversiones":[
        "dividendos",
        "intereses",
        "rendimientos",
        "acciones",
        "fondos",
        "ETF",
        "CDT",
        "inversión",
        "inversion"
    ],

    "Subsidios":[
        "subsidio",
        "beca",
        "ayuda económica",
        "ayuda economica",
        "reembolso",
        "auxilio",
        "incentivo",
        "apoyo económico"
    ],

    "Otros_Ingresos":[
        "premio",
        "propina",
        "regalo",
        "donación",
        "donacion",
        "regalías",
        "regalias",
        "ingreso ocasional"
    ]

}

# =====================================
# GENERAR DATASET
# =====================================

for tipo, conceptos in categorias.items():

    for _ in range(1500):

        descripcion = " ".join([

            random.choice(acciones),

            random.choice(["de", "por", ""]),

            random.choice(conceptos),

            random.choice(empresas),

            random.choice(bancos),

            random.choice(meses)

        ])

        descripcion = " ".join(descripcion.split())

        datos.append([
            descripcion,
            tipo
        ])

# =====================================
# DATAFRAME
# =====================================

df_train = pd.DataFrame(
    datos,
    columns=[
        "descripcion",
        "tipo_ingreso"
    ]
)

# Eliminar duplicados
df_train = df_train.drop_duplicates()

# Mezclar
df_train = df_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df_train.shape)
print(df_train.head())

(8966, 2)
                                         descripcion             tipo_ingreso
0             Ingreso acciones Microsoft BBVA agosto  Rendimiento_inversiones
1    Recepción de auxilio Empresa ABC BBVA diciembre                Subsidios
2  Cobro proyecto freelance Startup Banco Popular...               Honorarios
3             Consignación por inversión Empresa XYZ  Rendimiento_inversiones
4          Recepción de prima Google Daviplata abril                  Salario


In [2]:
# ============================================
# LIBRERÍAS
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# ============================================
# VARIABLES
# ============================================

X = df_train["descripcion"]
y = df_train["tipo_ingreso"]

# ============================================
# DIVISIÓN DEL DATASET
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================
# MODELO
# ============================================

modelo_tipo_ingreso = Pipeline([

    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1,2),
            min_df=2,
            max_df=0.95
        )
    ),

    (
        "clasificador",
        LogisticRegression(
            random_state=42,
            max_iter=1000,
            C=2
        )
    )

])

# ============================================
# ENTRENAMIENTO
# ============================================

modelo_tipo_ingreso.fit(X_train, y_train)

# ============================================
# PREDICCIONES
# ============================================

predicciones = modelo_tipo_ingreso.predict(X_test)

# ============================================
# MÉTRICAS
# ============================================

print("="*60)
print("EXACTITUD")
print("="*60)

print(accuracy_score(y_test, predicciones))

print()

print("="*60)
print("REPORTE DE CLASIFICACIÓN")
print("="*60)

print(classification_report(y_test, predicciones))

print()

print("="*60)
print("MATRIZ DE CONFUSIÓN")
print("="*60)

print(confusion_matrix(y_test, predicciones))

# ============================================
# GUARDAR MODELO
# ============================================

joblib.dump(
    modelo_tipo_ingreso,
    "modelos/modelo_tipo_ingreso_1.pkl"
)

print("\nModelo guardado correctamente.")

EXACTITUD
1.0

REPORTE DE CLASIFICACIÓN
                         precision    recall  f1-score   support

             Honorarios       1.00      1.00      1.00       299
                Negocio       1.00      1.00      1.00       300
         Otros_Ingresos       1.00      1.00      1.00       298
Rendimiento_inversiones       1.00      1.00      1.00       299
                Salario       1.00      1.00      1.00       299
              Subsidios       1.00      1.00      1.00       299

               accuracy                           1.00      1794
              macro avg       1.00      1.00      1.00      1794
           weighted avg       1.00      1.00      1.00      1794


MATRIZ DE CONFUSIÓN
[[299   0   0   0   0   0]
 [  0 300   0   0   0   0]
 [  0   0 298   0   0   0]
 [  0   0   0 299   0   0]
 [  0   0   0   0 299   0]
 [  0   0   0   0   0 299]]

Modelo guardado correctamente.


In [3]:
# ============================================
# PRUEBAS CON CONFIANZA
# ============================================

pruebas = [
    "Salario Bancolombia julio",
    "Pago honorarios consultoría",
    "Cobro factura cliente",
    "Dividendos de acciones",
    "Beca universitaria",
    "Premio concurso"
]

for texto in pruebas:

    # Predicción
    prediccion = modelo_tipo_ingreso.predict([texto])[0]

    # Probabilidades
    probabilidades = modelo_tipo_ingreso.predict_proba([texto])[0]

    # Confianza de la clase predicha
    confianza = probabilidades.max() * 100

    print(f"Descripción : {texto}")
    print(f"Predicción  : {prediccion}")
    print(f"Confianza   : {confianza:.2f}%")
    print("-" * 50)

Descripción : Salario Bancolombia julio
Predicción  : Salario
Confianza   : 95.40%
--------------------------------------------------
Descripción : Pago honorarios consultoría
Predicción  : Honorarios
Confianza   : 99.94%
--------------------------------------------------
Descripción : Cobro factura cliente
Predicción  : Negocio
Confianza   : 96.61%
--------------------------------------------------
Descripción : Dividendos de acciones
Predicción  : Rendimiento_inversiones
Confianza   : 99.98%
--------------------------------------------------
Descripción : Beca universitaria
Predicción  : Subsidios
Confianza   : 99.87%
--------------------------------------------------
Descripción : Premio concurso
Predicción  : Otros_Ingresos
Confianza   : 99.89%
--------------------------------------------------


# Modelo para tipo de ingreso (Salario o Independiente)

In [4]:
#librerias
# Modelo
#Python 3.12.2
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib
import pandas as pd

# --------------------------------------------
# DATASET DE ENTRENAMIENTO
# --------------------------------------------

datos_entrenamiento = {
    "descripcion": [

        # SALARIO
        "Sueldo recibido",
        "Salario depositado",
        "Deposoito de nomina",
        "Depósito de nómina",
        "Nomina recibida",
        "Nómina recibida",
        "Deposoito de sueldo",
        "Depósito de sueldo",
        "Sueldo mensual recibido",
        "Salario mensual acreditado",
        "Remuneración mensual",
        "Bonificación laboral",
        "Horas extras pagadas",
        "Utilidades recibidas",
        "Gratificación",
        "Jubilación recibida",

        # INDEPENDIENTE
        "Honorarios recibidos",
        "Pago recibido",
        "Transferencia entrante",
        "Abono recibido",
        "Cobro de cliente",
        "Factura cobrada",
        "Trabajo freelance cobrado",
        "Comisión ganada",
        "Ganancia recibida",
        "Ingreso por ventas",
        "Consultoría",
        "Cobro por consultoría",
        "Ingreso del negocio",
        "Renta recibida",
        "Alquiler cobrado",
        "Dividendos",
        "Intereses ganados",
        "Regalías",
        "Propinas recibidas",
        "Premio en efectivo"

    ],

    "tipo_ingreso": [

        "Salario","Salario","Salario","Salario","Salario","Salario",
        "Salario","Salario","Salario","Salario","Salario","Salario",
        "Salario","Salario","Salario","Salario",

        "Independiente","Independiente","Independiente","Independiente",
        "Independiente","Independiente","Independiente","Independiente",
        "Independiente","Independiente","Independiente","Independiente",
        "Independiente","Independiente","Independiente","Independiente",
        "Independiente","Independiente","Independiente","Independiente"

    ]
}

df_train = pd.DataFrame(datos_entrenamiento)

# --------------------------------------------
# VARIABLES
# --------------------------------------------

X = df_train["descripcion"]
y = df_train["tipo_ingreso"]

# --------------------------------------------
# DIVIDIR DATASET
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# --------------------------------------------
# MODELO
# --------------------------------------------

modelo = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True)),
    ("clasificador", LogisticRegression(random_state=42))
])

# --------------------------------------------
# ENTRENAMIENTO
# --------------------------------------------

modelo.fit(X_train, y_train)

# --------------------------------------------
# EVALUACIÓN
# --------------------------------------------

pred = modelo.predict(X_test)

joblib.dump(modelo, "modelos/modelo_tipo_ingreso_2.pkl")

print("Exactitud:", accuracy_score(y_test, pred))
print()
print(classification_report(y_test, pred))


Exactitud: 0.75

               precision    recall  f1-score   support

Independiente       0.75      1.00      0.86         6
      Salario       0.00      0.00      0.00         2

     accuracy                           0.75         8
    macro avg       0.38      0.50      0.43         8
 weighted avg       0.56      0.75      0.64         8



c:\Users\USUARIO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USUARIO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USUARIO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

# Modelo Perfil financiero

In [5]:
df2 = pd.read_csv("datasets/df_modelo_pf.csv", encoding="utf-8-sig")
pd.set_option('display.max_columns', None)
df2

,fecha,id_usuario,ingreso_mensual,gasto_mensual_total,tasa_ahorro,objetivo_presupuesto,escenario_financiero,puntaje_crediticio,relacion_deuda_ingreso,pago_prestamo,monto_inversion,servicios_suscripcion,fondo_emergencia,cantidad_transacciones,indicador_fraude,gastos_discrecionales,gastos_esenciales,tipo_ingreso,alquiler_o_hipoteca,categoria,estado_flujo_caja,puntaje_recomendacion_financiera,nivel_estres_financiero,ahorro_real,meta_ahorro_cumplida,indice_financiero,perfil_financiero
0,2019-01-01,1584,3119.58,3212.07,0.38,3676.11,Inflación,721.0,0.56,125.77,689.22,3,510.58,68,0,857.55,1910.85,Independiente,1501.65,Inversiones,Positivo,8.3,Bajo,0.00,0,50.345998,En observación
1,2019-01-31,1045,3262.44,3732.81,0.10,2607.17,Inflación,670.0,0.42,454.19,360.34,4,1154.41,41,0,534.51,3165.20,Salario,1603.17,Inversiones,Positivo,22.6,Bajo,0.00,0,39.294951,Crítico
2,2019-03-02,1756,2931.20,3335.58,0.15,3004.14,Inflación,691.0,0.24,971.82,0.00,5,1433.02,90,0,353.67,1504.56,Independiente,1097.82,Salud,Positivo,58.8,Bajo,0.00,0,47.056776,En observación
3,2019-04-01,1724,3506.79,2327.59,0.17,3346.97,Normal,717.0,0.16,482.76,182.06,5,227.37,94,0,594.08,1450.72,Independiente,1155.64,Alimentacion,Positivo,74.5,Bajo,1179.20,0,53.087863,Estable
4,2019-05-01,1600,4606.87,2182.58,0.34,2670.09,Inflación,795.0,0.25,263.74,342.78,9,589.81,73,0,556.86,1000.00,Salario,1170.86,Servicios,Negativo,38.7,Alto,2424.29,0,61.782912,Saludable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2023-07-09,1822,3254.55,3992.02,0.17,2675.77,Normal,704.0,0.30,285.16,597.74,9,466.11,81,0,408.23,2088.32,Mixto,2079.99,Salud,Positivo,43.2,Bajo,0.00,0,47.085864,En observación
2996,2023-08-08,1907,2752.78,5685.18,0.26,2682.99,Recesión,697.0,0.12,417.06,587.44,7,807.86,21,0,167.78,2445.34,Salario,832.95,Ocio,Neutral,46.0,Bajo,0.00,0,56.078956,Estable
2997,2023-09-07,1464,3691.10,3228.05,0.12,3893.53,Normal,626.0,0.12,621.01,286.47,6,780.17,34,0,275.53,3231.56,Salario,790.68,Servicios,Neutral,40.7,Medio,463.05,0,49.541143,En observación
2998,2023-10-07,1346,3133.24,4335.72,0.08,2854.72,Inflación,628.0,0.42,650.23,966.60,2,712.12,92,0,373.85,2305.14,Salario,903.82,Servicios,Positivo,57.1,Bajo,0.00,0,41.270962,En riesgo


In [6]:
# Seleccionar únicamente las columnas que utilizará el modelo
columnas_modelo = [

    "ingreso_mensual",
    "gasto_mensual_total",
    "tasa_ahorro",
    "objetivo_presupuesto",
    "relacion_deuda_ingreso",
    "pago_prestamo",
    "monto_inversion",
    "servicios_suscripcion",
    "fondo_emergencia",
    "cantidad_transacciones",
    "gastos_discrecionales",
    "gastos_esenciales",
    "tipo_ingreso",
    "alquiler_o_hipoteca",
    "estado_flujo_caja",
    "nivel_estres_financiero",
    "ahorro_real",
    "perfil_financiero"
]

# Crear el dataframe para entrenamiento
df_modelo = df2[columnas_modelo].copy()

# ==========================================================
# VERIFICAR VALORES NULOS
# ==========================================================

print("Valores nulos por columna:")
print(df_modelo.isnull().sum())

# ==========================================================
# ELIMINAR FILAS CON NULOS (si existen)
# ==========================================================

df_modelo = df_modelo.dropna()

# ==========================================================
# SEPARAR VARIABLES DE ENTRADA Y VARIABLE OBJETIVO
# ==========================================================

X = df_modelo.drop(columns="perfil_financiero")
y = df_modelo["perfil_financiero"]

# ==========================================================
# MOSTRAR INFORMACIÓN DEL DATASET
# ==========================================================

print("\nColumnas numéricas:")
print(X.select_dtypes(exclude="object").columns.tolist())

print("\nColumnas categóricas:")
print(X.select_dtypes(include="object").columns.tolist())

print("\nDistribución del perfil financiero:")
print(y.value_counts())

print("\nDimensiones del conjunto de datos:")
print("X:", X.shape)
print("y:", y.shape)
# ==========================================================
# LIBRERÍAS
# ==========================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ==========================================================
# COLUMNAS
# ==========================================================

categoricas = X.select_dtypes(include="object").columns.tolist()
numericas = X.select_dtypes(exclude="object").columns.tolist()

# ==========================================================
# PREPROCESAMIENTO
# ==========================================================

preprocesador = ColumnTransformer(

    transformers=[

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categoricas
        ),

        (
            "num",
            "passthrough",
            numericas
        )

    ]

)

# ==========================================================
# MODELO
# ==========================================================

modelo = Pipeline([

    ("preprocesamiento", preprocesador),

    ("clasificador",
     RandomForestClassifier(
         n_estimators=300,
         random_state=42,
         class_weight="balanced"
     )
    )

])

# ==========================================================
# DIVISIÓN DEL DATASET
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

# ==========================================================
# ENTRENAMIENTO
# ==========================================================

modelo.fit(X_train, y_train)

# ==========================================================
# PREDICCIÓN
# ==========================================================

predicciones = modelo.predict(X_test)

# ==========================================================
# MÉTRICAS
# ==========================================================

print("Accuracy:", accuracy_score(y_test, predicciones))

print("\n")

print(classification_report(y_test, predicciones))

print("\nMatriz de confusión:\n")

print(confusion_matrix(y_test, predicciones))

Valores nulos por columna:
ingreso_mensual            0
gasto_mensual_total        0
tasa_ahorro                0
objetivo_presupuesto       0
relacion_deuda_ingreso     0
pago_prestamo              0
monto_inversion            0
servicios_suscripcion      0
fondo_emergencia           0
cantidad_transacciones     0
gastos_discrecionales      0
gastos_esenciales          0
tipo_ingreso               0
alquiler_o_hipoteca        0
estado_flujo_caja          0
nivel_estres_financiero    0
ahorro_real                0
perfil_financiero          0
dtype: int64

Columnas numéricas:
['ingreso_mensual', 'gasto_mensual_total', 'tasa_ahorro', 'objetivo_presupuesto', 'relacion_deuda_ingreso', 'pago_prestamo', 'monto_inversion', 'servicios_suscripcion', 'fondo_emergencia', 'cantidad_transacciones', 'gastos_discrecionales', 'gastos_esenciales', 'alquiler_o_hipoteca', 'ahorro_real']

Columnas categóricas:
['tipo_ingreso', 'estado_flujo_caja', 'nivel_estres_financiero']

Distribución del perfil finan

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_9236\1858627312.py:55: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(X.select_dtypes(include="object").columns.tolist())
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_9236\1858627312.py:82: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user

Accuracy: 0.6216666666666667


                precision    recall  f1-score   support

       Crítico       0.78      0.78      0.78        60
En observación       0.61      0.63      0.62       150
     En riesgo       0.68      0.64      0.66       120
       Estable       0.50      0.53      0.52       120
     Excelente       0.74      0.77      0.75        60
     Saludable       0.55      0.49      0.52        90

      accuracy                           0.62       600
     macro avg       0.64      0.64      0.64       600
  weighted avg       0.62      0.62      0.62       600


Matriz de confusión:

[[47  0 13  0  0  0]
 [ 0 95 22 32  0  1]
 [13 30 77  0  0  0]
 [ 0 32  2 64  0 22]
 [ 0  0  0  1 46 13]
 [ 0  0  0 30 16 44]]


In [7]:
joblib.dump( modelo,
    "modelos/perfil_financiero.pkl"
)
print("\nModelo guardado correctamente.")


Modelo guardado correctamente.
